# LangGraph G5 — Short-term memory and context management
CampusAI v3 forgets everything between invocations. LangGraph solves this with a
**checkpointer**: after every node the whole state is saved under a **thread id**. Invoking the
same thread again loads the saved state first.

```text
thread "rahul":   run 1 -> checkpoint -> run 2 -> checkpoint -> run 3 ...
thread "priya":   run 1 -> checkpoint ...              (separate, never mixed)
```

This is more than chat memory. A checkpoint after every node means a crashed run can resume at
the node where it died, and a paused run (G8) can wait for a human for hours. LangGraph calls
this **durable execution**. Here the checkpointer keeps everything in RAM; SQLite and Postgres
checkpointers have the same interface.

Memory creates a second problem: threads grow, and every message is sent to the model on every
turn. **Context management** keeps the context small: summarise old turns, keep recent ones.

### Step 1 — Compile with a checkpointer, invoke with a thread id

> **Why LangGraph has a *checkpointer* and *threads***
>
> Saving the state only at the end would give you chat memory. Saving it after every node is what makes a run resumable at the exact node where it paused or crashed, which G8 and G14 depend on. A thread id groups those checkpoints into one conversation, so many conversations can share one graph.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver     # LangGraph: saves the state after every node

campusai_v4 = build_desks(checkpointer=InMemorySaver())    # LangGraph: persistence is a compile-time option

rahul = {"configurable": {"thread_id": "rahul-1"}}          # LangGraph: the run config; thread_id names the conversation
out = campusai_v4.invoke({"messages": [HumanMessage("My name is Rahul and my student id is S001. What is my attendance?")]}, rahul)
print("turn 1 :", text_of(out["messages"][-1])[:110])
out = campusai_v4.invoke({"messages": [HumanMessage("What is my name?")]}, rahul)
print("turn 2 :", text_of(out["messages"][-1])[:110])
print("messages on this thread:", len(out["messages"]))

priya = {"configurable": {"thread_id": "priya-1"}}
out = campusai_v4.invoke({"messages": [HumanMessage("What is my name?")]}, priya)
print("other thread:", text_of(out["messages"][-1])[:110])

> **What just happened**
>
> Turn 1 on thread rahul-1 ran the desk graph and, because a checkpointer was compiled in, the state after every node was saved under that thread id. Turn 2 started by loading that saved state, so the new question was appended after the earlier messages and the model could see the name; the count of 7 is turn 1's conversation plus turn 2. Thread priya-1 had no saved state, so its run started empty and the model did not know the name.

### Step 2 — Inspect the saved state and its history

`get_state()` reads the latest checkpoint without running anything. `get_state_history()`
lists every checkpoint of the thread: which node ran next, and what the state was. This is the
audit trail, and the basis for resuming after a crash or "time travelling" while debugging.

In [ ]:
snapshot = campusai_v4.get_state(rahul)                     # LangGraph: latest checkpoint of the thread
print("latest state has", len(snapshot.values["messages"]), "messages; next node to run:", snapshot.next or "(none, run finished)")

history = list(campusai_v4.get_state_history(rahul))        # LangGraph: every checkpoint, newest first
print("checkpoints on thread rahul-1:", len(history))
for snap in list(reversed(history))[:6]:                    # oldest first
    print(f"  next={str(snap.next):14} messages={len(snap.values.get('messages', []))}")

> **What just happened**
>
> No node ran; these calls only read checkpoints. `next` is empty because the last run finished. The history is one row per saved checkpoint, oldest first: `__start__` with 0 messages is the moment before turn 1 began, then `triage`, then `records`, then an empty `next` when turn 1 completed, then turn 2 starting again. Each row is a point the run could be resumed from.

### Step 3 — Keep the context small: summarise old turns

A `manage_context` node runs before the agent. When the thread is long it asks the model for a
summary of the older messages, stores it in the state, and deletes those messages with
`RemoveMessage` (the `add_messages` reducer understands deletions). The agent node injects the
summary into its system prompt, so nothing important is lost and the context stays bounded.

In [ ]:
class MemoryState(TypedDict, total=False):                 # ours: ChatState plus a running summary
    messages: Annotated[list, add_messages]
    summary: str

KEEP_RECENT = 4                                            # ours: how many recent messages stay verbatim

def manage_context(state: MemoryState):                    # ours: runs before the agent on every turn
    messages = state["messages"]
    if len(messages) <= KEEP_RECENT + 2:
        return {}
    old, recent = messages[:-KEEP_RECENT], messages[-KEEP_RECENT:]
    prompt = "Summarise the conversation so far in two sentences, keeping names, ids and decisions." + (f" Previous summary: {state['summary']}" if state.get("summary") else "")
    summary = model.invoke([SystemMessage(prompt)] + old)  # LangChain
    return {"summary": text_of(summary), "messages": [RemoveMessage(id=m.id) for m in old]}   # LangChain RemoveMessage; the reducer deletes them

def agent_with_summary(state: MemoryState):                # ours
    persona = CAMPUS_PERSONA + (f" Summary of earlier conversation: {state['summary']}" if state.get("summary") else "")
    reply = model.bind_tools(READ_TOOLS).invoke([SystemMessage(persona)] + state["messages"])   # LangChain
    return {"messages": [reply]}

g = StateGraph(MemoryState)
g.add_node("manage_context", manage_context)
g.add_node("agent", agent_with_summary)
g.add_node("tools", ToolNode(READ_TOOLS))                  # LangGraph
g.add_edge(START, "manage_context")
g.add_edge("manage_context", "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
campusai_v5 = g.compile(checkpointer=InMemorySaver())

thread = {"configurable": {"thread_id": "long-1"}}
for text in ["My name is Rahul.", "Look up student S001.", "And course CS201?", "What is the weather like?", "What is my name?"]:
    out = campusai_v5.invoke({"messages": [HumanMessage(text)]}, thread)
    print(f"{text:26} -> {len(out['messages']):2} messages kept | summary: {(out.get('summary') or '-')[:70]}")
print("\nlast answer:", text_of(out["messages"][-1])[:100])

> **What just happened**
>
> Every turn ran START -> `manage_context` -> `agent` (-> `tools` -> `agent`). For the first turns the list was short, so `manage_context` returned nothing. Once the list passed the threshold it asked the model for a summary, stored it in the state, and returned RemoveMessage updates for the old messages; the reducer deleted them, which is why the count drops and then rises again. On the last turn the name was no longer in the messages, but the agent node had put the summary into its system prompt, so the model still answered correctly.

### Recap

- **The problem we started with:** every invocation started from an empty state, and a remembered thread grows without limit.
- **What we added:** a checkpointer with thread ids, `get_state` and `get_state_history`, and a context-management node using `RemoveMessage` plus a summary.
- **What you saw in the output:** turn 2 answered from turn 1; the long thread stayed at a handful of messages while the summary carried the facts.
- **Carry forward:** G6 adds memory that survives across threads: facts about the person, not the conversation.